# C2.5 · Supply-chain research

**Function C — Offensive Security & Research → The Security Researcher**  ·  *Security of AI*

---

**Risk.** Adapter and LoRA provenance, registry tampering, dependency confusion in agent ecosystems.

**Control.** Verify provenance; sign and attest artefacts.

**This lab.** Detect a tampered adapter through provenance.

| | |
|---|---|
| Open-source tooling | Sigstore, in-toto, OWASP AIBOM |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C2.5"))

Supply-chain research for AI systems has the same shape as for software, plus two new artefacts nobody has a process for: model weights and prompt/tool packages.

In [ ]:
from cybercommons import research
P = research.Package

candidates = [
    P("requests",  "2.31.0", signed=True,  downloads=900_000, age_days=400),
    P("requsts",   "2.31.0", signed=False, downloads=12,      age_days=3),
    P("colorama",  "0.4.6",  signed=True,  downloads=500_000, age_days=900),
    P("colourama", "0.4.6",  signed=False, downloads=40,      age_days=9),
    P("mcp-github-tools", "0.1.0", signed=False, downloads=200, age_days=11),
]
for p in candidates:
    r = research.provenance(p)
    print(f"{r['package']:26s} {r['verdict']:7s}")
    for f in r["flags"]:
        print(f"      · {f}")

The last entry is the new shape: an MCP tool package, unsigned and days old, that would run inside your agent with its authority. It trips the same signals — which is good news, because it means the existing process extends rather than needing invention.

### Expect

The two legitimate packages are allowed. Both typosquats are blocked with the distance and the popular name they imitate. The MCP package lands on review or block for being unsigned, new and unscrutinised.

### Your turn

Add the model-weight case: what are the equivalents of 'signed' and 'downloads' for a checkpoint? Sigstore covers the first. The second has no good answer yet — say so in your report.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C2.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*